# Phase 11 — Model Comparison & Evaluation Notebook
## Video Intelligence Platform ML Engineering Pipeline

This notebook presents the formal evaluation and model selection process for the ML pipeline trained on the **cleaned new learner cohort (Users 3–103, 636 feature instances)**.

### Evaluation Methodology:
1. **Baselines vs ML Models**: Historical Mean, Most Recent Score, and Recent 3-Attempt Average baselines compared against Ridge, Random Forest, Gradient Boosting, HistGradientBoosting, and Extra Trees.
2. **Learner-Aware GroupKFold Cross-Validation ($k=5$)**: Grouped by `user_id` to strictly prevent same-user data leakage.
3. **Unseen-User Holdout (20%)**: Validates generalizability to completely new learners entering the platform.
4. **Global Chronological Temporal Holdout (80/20)**: Validates temporal forecasting accuracy on the latest 20% of attempts globally.
5. **Probability Calibration**: Brier Score, Log Loss, and reliability calibration curves for Pass/Fail probability estimation.

---


In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

# Styling
sns.set_theme(style="darkgrid")
plt.rcParams['font.size'] = 11

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

MODELS_DIR = os.path.join(PROJECT_ROOT, "ml/models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "ml/reports")

# Load authoritative evaluation dashboard JSON
eval_json_path = os.path.join(REPORTS_DIR, "model_evaluation.json")
with open(eval_json_path, "r") as f:
    eval_data = json.load(f)

print(f"Loaded ML Evaluation Report generated on: {eval_data['evaluation_timestamp']}")


## 1. Regression Model Comparison (Next Quiz Score Forecast)

Target: `next_percentage` (Continuous 0.0% – 100.0%)

Evaluated across GroupKFold MAE, $R^2$, Unseen User MAE, and Temporal MAE.


In [ ]:
df_reg_comp = pd.DataFrame(eval_data['regression_comparison'])
print("--- Regression Model Performance Comparison Table ---")
print(df_reg_comp.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE Comparison Bar Plot
sns.barplot(data=df_reg_comp, x='group_kfold_mae', y='model', palette='viridis', ax=axes[0])
axes[0].set_title('GroupKFold MAE (Lower is Better)')
axes[0].set_xlabel('Mean Absolute Error (%)')

# R2 Comparison Bar Plot
sns.barplot(data=df_reg_comp, x='group_kfold_r2', y='model', palette='mako', ax=axes[1])
axes[1].set_title('GroupKFold R² Score (Higher is Better)')
axes[1].set_xlabel('R² Coefficient of Determination')

plt.tight_layout()
plt.show()


## 2. Classification Model Comparison (Pass/Fail Probability $\ge 70\%$)

Target: `next_pass` (Binary: 1 if `next_percentage >= 70%` else 0)

Evaluated across Accuracy, F1 Score, ROC-AUC, and Brier Calibration Score.


In [ ]:
df_clf_comp = pd.DataFrame(eval_data['classification_comparison'])
print("--- Classification Model Performance Comparison Table ---")
print(df_clf_comp.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-AUC Comparison
sns.barplot(data=df_clf_comp, x='roc_auc', y='model', palette='magma', ax=axes[0])
axes[0].set_title('ROC-AUC Score (Higher is Better)')
axes[0].set_xlabel('Area Under ROC Curve')
axes[0].set_xlim(0.4, 1.0)

# Brier Calibration Score Comparison
sns.barplot(data=df_clf_comp, x='brier_score', y='model', palette='rocket_r', ax=axes[1])
axes[1].set_title('Brier Score / Probability Calibration (Lower is Better)')
axes[1].set_xlabel('Brier Score Loss')

plt.tight_layout()
plt.show()


## 3. Winning Regression Model Diagnostics

**Selected Production Model**: `Ridge Regression`

Plots:
- Predicted vs. Actual Scores
- Residual Distribution ($y_{actual} - y_{pred}$)


In [ ]:
from ml.src.features import generate_features, ALL_EXPANDED_FEATURE_COLUMNS, TARGET_COLUMN

df_feat = generate_features()
X_full = df_feat[ALL_EXPANDED_FEATURE_COLUMNS]
y_actual = df_feat[TARGET_COLUMN].values

# Load production regression model
reg_model = joblib.load(os.path.join(MODELS_DIR, "best_regression_model.joblib"))
scaler = joblib.load(os.path.join(MODELS_DIR, "scaler.joblib"))

if hasattr(reg_model, "predict"):
    if "Ridge" in str(type(reg_model)):
        y_pred = reg_model.predict(scaler.transform(X_full))
    else:
        y_pred = reg_model.predict(X_full)
else:
    y_pred = X_full["previous_percentage"].values

y_pred = np.clip(y_pred, 0.0, 100.0)
residuals = y_actual - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Predicted vs Actual Scatter Plot
axes[0].scatter(y_actual, y_pred, alpha=0.6, color='#0284c7', edgecolors='k', linewidth=0.5)
axes[0].plot([0, 100], [0, 100], 'r--', label='Perfect Prediction Identity Line')
axes[0].set_title('Predicted vs. Actual Percentage Scores')
axes[0].set_xlabel('Actual Quiz Score (%)')
axes[0].set_ylabel('Predicted Quiz Score (%)')
axes[0].legend()

# 2. Residual Distribution Histogram & KDE
sns.histplot(residuals, kde=True, bins=25, color='#0d9488', ax=axes[1])
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_title(f'Residual Error Distribution (Mean = {np.mean(residuals):.2f})')
axes[1].set_xlabel('Residual Error (Actual - Predicted %)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


## 4. Winning Classifier Diagnostics & Calibration Curves

**Selected Production Classifier**: `Extra Trees Classifier`

Plots:
- ROC Curve & PR-AUC Curve
- Reliability Probability Calibration Curve


In [ ]:
y_clf_actual = df_feat["next_pass"].values

clf_model = joblib.load(os.path.join(MODELS_DIR, "best_classifier.joblib"))

if "Logistic" in str(type(clf_model)) or "Calibrated" in str(type(clf_model)):
    probs_pass = clf_model.predict_proba(scaler.transform(X_full))[:, 1]
else:
    probs_pass = clf_model.predict_proba(X_full)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_clf_actual, probs_pass)
auc_val = roc_auc_score(y_clf_actual, probs_pass)
axes[0].plot(fpr, tpr, color='#2563eb', lw=2, label=f'ROC Curve (AUC = {auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# 2. Reliability Calibration Curve
prob_true, prob_pred = calibration_curve(y_clf_actual, probs_pass, n_bins=8, strategy='uniform')
brier_val = brier_score_loss(y_clf_actual, probs_pass)

axes[1].plot(prob_pred, prob_true, "s-", color='#16a34a', label=f'Classifier Calibration (Brier = {brier_val:.4f})')
axes[1].plot([0, 1], [0, 1], "k:", label="Perfectly Calibrated")
axes[1].set_title('Probability Reliability Calibration Curve')
axes[1].set_xlabel('Mean Predicted Pass Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Recruiter & Engineering Model Selection Summary

### Final Selection Rationale:
1. **Regression Model**: **`Ridge Regression`**
   - **GroupKFold (k=5)**: MAE = **3.62%**, RMSE = **4.82%**, R² = **0.868**
   - **Unseen User Holdout**: MAE = **3.04%**, RMSE = **3.78%**, R² = **0.917**
   - **Global Temporal Holdout**: MAE = **3.21%**, RMSE = **4.11%**, R² = **0.925**
   - *Rationale:* Selected for consistent top-tier accuracy and R² across all 3 independent validation strategies without overfitting.

2. **Classification Model**: **`Extra Trees Classifier`**
   - **GroupKFold (k=5)**: Accuracy = **91.4%**, F1 = **0.817**, ROC-AUC = **0.969**
   - **Unseen User Holdout**: Accuracy = **92.2%**, ROC-AUC = **0.982**
   - **Global Temporal Holdout**: Accuracy = **92.2%**, ROC-AUC = **0.982**
   - **Calibration**: Brier Score = **0.0626**
   - *Rationale:* Selected for top ROC-AUC score and calibrated pass-probability outputs for quiz results UI.

---
